In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Assumes the notebook's working directory is notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "loan_default_dataset.csv"

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()

# Remove the blank column if it is an unwanted exported index
df = df.loc[:, df.columns != ""]

target_column = "DEFAULT_STATUS"

X = df.drop(columns=[target_column, "CUSTOMER_ID"])
y = df[target_column]

print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(y.value_counts())

if y.nunique() < 2 or y.value_counts().min() < 2:
    raise ValueError(
        "Insufficient examples per class for a stratified train/test split."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining set:", X_train.shape)
print("Test set:", X_test.shape)

Dataset shape: (10, 14)

Target distribution:
DEFAULT_STATUS
0    6
1    4
Name: count, dtype: int64

Training set: (8, 12)
Test set: (2, 12)


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

numerical_cols = X_train.select_dtypes(include="number").columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

def create_preprocessor():
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numerical_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42
    )
}

results = []
trained_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", create_preprocessor()),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        )
    })

    trained_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    by="F1 Score", ascending=False
)

display(results_df)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,1.0,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0,1.0
2,XGBoost,0.5,0.0,0.0,0.0


In [4]:
comparison = pd.DataFrame({
    "Actual": y_test.to_numpy()
})

for name, pipeline in trained_pipelines.items():
    comparison[name] = pipeline.predict(X_test)

print("Actual vs predicted:")
display(comparison)

Actual vs predicted:


,Actual,Logistic Regression,Random Forest,XGBoost
0,0,0,0,0
1,1,1,1,0


In [5]:
from pathlib import Path
import joblib

# Find the project root (assuming this notebook runs from notebooks/)
PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Select candidate model
best_model_name = "Logistic Regression"
best_pipeline = trained_pipelines[best_model_name]

# Save the complete pipeline
model_path = MODEL_DIR / "loan_default_pipeline.joblib"
joblib.dump(best_pipeline, model_path)

print("Selected model:", best_model_name)
print("Saved model to:", model_path.resolve())
print("File exists:", model_path.exists())

Selected model: Logistic Regression
Saved model to: E:\BW\ML projects\loan_default_prediction\models\loan_default_pipeline.joblib
File exists: True


In [6]:
loaded_model = joblib.load(model_path)

predictions = loaded_model.predict(X_test)

print("Actual values:    ", y_test.to_numpy())
print("Loaded predictions:", predictions)
print("Model loaded successfully!")

Actual values:     [0 1]
Loaded predictions: [0 1]
Model loaded successfully!
